# 06. Tuning de Hiperparâmetros: HalvingRandomSearchCV


Nesta etapa, focaremos na melhoria da Performance do Modelo através do ajuste fino de hiperparâmetros, saindo da configuração padrão (Baseline).

Para equilibrar a precisão da busca com o Custo Operacional, abandonaremos o GridSearch tradicional em favor do HalvingRandomSearchCV. Esta técnica utiliza uma abordagem de "Torneio":

Fase 1: Testa muitas combinações aleatórias com poucos dados (baixo custo/rápido).

Fase 2: Elimina os piores modelos (fail-fast).

Fase 3: Re-treina os melhores candidatos com progressivamente mais dados.

Isso garante que o poder computacional seja gasto apenas refinando as configurações que demonstram real potencial de reduzir o erro (RMSE), otimizando o tempo de entrega do modelo final.

## 🛠️  Imports e Configurações

### Pacotes

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))



In [2]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import warnings

from sklearn.experimental import enable_halving_search_cv 
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, HalvingRandomSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

from src.tools import avaliar_modelo, registrar_metricas, verificar_features, excluir_colunas_modelo, load_config

### Configurações gerais

In [3]:
plt.rcParams['figure.figsize'] = (12, 6)
warnings.filterwarnings("ignore")
PATH_DATA = '../models/model_final.pkl'     # notebook 05
DATA_XTRAIN = '../data/processed/X_train_processed.csv'  #colunas + treino
DATA_YTRAIN = '../data/processed/y_train.csv'            #colunas
DATA_XVAL = '../data/processed/X_val_processed.csv'      #colunas + treino
DATA_YVAL = '../data/processed/y_val.csv'                #colunas


In [4]:
#🚩
config = load_config()
EXCLUIR_COLUNA_MODELO = config['excluir_colunas_modelo']
print(f"Colunas excluidas: {EXCLUIR_COLUNA_MODELO}")  


Colunas excluidas: ['date', 'sales', 'id', 'sales_log', 'day_name', 'holiday_name', 'year']


## ⛁ Carregando base

In [5]:
# Carrega o modelo
model1 = joblib.load(PATH_DATA)
colunas_treino = model1.feature_names_in_

# verifica se o modelo que esta sendo utilizado e o mesmo que foi processado anteriormente
#🚩
# verificar_features(colunas_treino, cols_to_drop)
print (f'Colunas do modelo: {len(colunas_treino)}: {colunas_treino} ')


Colunas do modelo: 30: ['is_weekend' 'is_payday' 'is_holiday' 'days_until_holiday' 'month_sin'
 'month_cos' 'day_of_week_sin' 'day_of_week_cos' 'day_of_year_sin'
 'day_of_year_cos' 'lag_1' 'lag_2' 'lag_3' 'lag_7' 'lag_14' 'lag_21'
 'lag_28' 'lag_91' 'rolling_mean_7' 'rolling_std_7' 'rolling_mean_28'
 'rolling_std_28' 'rolling_mean_91' 'rolling_std_91' 'store' 'item'
 'month' 'day' 'day_of_week' 'day_of_year'] 


In [6]:
#🚩
#  >>>Validação de  estrutura de colunas para o modelo

# Carregamento dos dados brutos
X_train_log = pd.read_csv(DATA_XTRAIN)  
X_val_log = pd.read_csv(DATA_XVAL)   
y_train_log = pd.read_csv(DATA_YTRAIN)    
y_val_log = pd.read_csv(DATA_YVAL)     

X_train_log = excluir_colunas_modelo(X_train_log, EXCLUIR_COLUNA_MODELO)
X_val_log = excluir_colunas_modelo(X_val_log, EXCLUIR_COLUNA_MODELO)
# Não aplica a 'y_train_log', 'y_val_log' -so tem o target

# Transformação: Converte para Array 1D (formato exigido pelo modelo)
y_train_log = y_train_log.values.ravel()
y_val_log = y_val_log.values.ravel()

print(f"Histórico de Treino: {X_train_log.shape[0]} registros.")
print(f"Dados de Validação: {X_val_log.shape[0]} registros.")
print(f"\nColunas efetivas: {len(X_train_log.columns)}: {list(X_train_log.columns)}") # tem que ser igual ao do notebook 05

Histórico de Treino: 775500 registros.
Dados de Validação: 46000 registros.

Colunas efetivas: 30: ['is_weekend', 'is_payday', 'is_holiday', 'days_until_holiday', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_91', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28', 'rolling_mean_91', 'rolling_std_91', 'store', 'item', 'month', 'day', 'day_of_week', 'day_of_year']


## 🏋️‍♂️ Preparando para treinar o modelo

In [7]:
# >>>Estratégia de Validação (Respeitando o Tempo)

# Problema de Negócio: Prever os próximos 7 dias.
# Não podemos usar validação aleatória (K-Fold comum) pois o futuro não explica o passado.
# Usamos TimeSeriesSplit para simular janelas deslizantes de tempo.
tscv = TimeSeriesSplit(n_splits=3)   # vai treinar o modelo 3 vezes para cada combinação de parâmetros


In [ ]:
# >>>Definição do Modelo e Hiperparâmetros 


#----------------------------------------------------------------------/produção
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Grid de Hiperparâmetros focado nas Hipóteses:
# - n_estimators: Estabilidade da previsão.
# - max_depth: Capacidade de aprender padrões complexos (ex: Efeito Payday dia 1-10).
# - min_samples_leaf: Evitar overfitting (decorar vendas de um dia específico).

param_dist = {      
    'max_depth': [10, 15, 20],            # [10, 20, None] None deixa a árvore crescer para pegar picos de feriados
    'min_samples_split': [2, 5, 10],        
    'min_samples_leaf': [1, 2, 4],          
    'bootstrap': [True]
}


In [ ]:
# >>>Otimização (Tuning - HalvingRandomSearchCV )

# Métrica: 'neg_root_mean_squared_error' (RMSE).
# Motivo: Penaliza erros grandes. No varejo, um erro grande significa
# ruptura total (perda de venda) ou estoque parado (custo).

#valor real  ------------------------------------------------------------produção
halving_search = HalvingRandomSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    factor=3,
    resource='n_samples',
    max_resources=50000,     
    min_resources=2000, 
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)


# Treinamento
halving_search.fit(X_train_log, y_train_log)


n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 2000
max_resources_: 50000
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 25
n_resources: 2000
Fitting 3 folds for each of 25 candidates, totalling 75 fits
----------
iter: 1
n_candidates: 9
n_resources: 6000
Fitting 3 folds for each of 9 candidates, totalling 27 fits
----------
iter: 2
n_candidates: 3
n_resources: 18000
Fitting 3 folds for each of 3 candidates, totalling 9 fits


,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'bootstrap': [True], 'max_depth': [10, 15, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...]}"
,n_candidates,'exhaust'
,factor,3
,resource,'n_samples'
,max_resources,50000
,min_resources,2000
,aggressive_elimination,False
,cv,TimeSeriesSpl...est_size=None)
,scoring,'neg_root_mean_squared_error'
,refit,True


In [10]:
# >>>Avaliação do Melhor Modelo     

best_model = halving_search.best_estimator_

print("\n--- Configuração Campeã ---")
print(halving_search.best_params_)

# Previsão nos dados de validação (ainda em escala Log, se o treino foi Log)
y_pred_log = best_model.predict(X_val_log)


# Reverte o Log para Vendas Reais (np.expm1)
# IMPORTANTE: Assume que y_train estava em log (np.log1p) feito no notebook anterior
y_pred_real = np.expm1(y_pred_log) 
y_val_real = np.expm1(y_val_log) 

# Trava de segurança: Vendas não podem ser negativas
y_pred_real = np.maximum(y_pred_real, 0)



--- Configuração Campeã ---
{'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 10, 'bootstrap': True}


In [11]:
# Logo após definir o best_model ou rodar o fit
print("Quantidade de colunas usadas:", best_model.n_features_in_)
print("\nLista EXATA de colunas no modelo:")
print(list(best_model.feature_names_in_))


Quantidade de colunas usadas: 30

Lista EXATA de colunas no modelo:
['is_weekend', 'is_payday', 'is_holiday', 'days_until_holiday', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_91', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28', 'rolling_mean_91', 'rolling_std_91', 'store', 'item', 'month', 'day', 'day_of_week', 'day_of_year']


In [12]:
max_val = max(y_val_log.max(), y_pred_log.max())
if max_val > 15: # 15 em log é ~3.2 milhões. Seguro para varejo.
    raise ValueError(f"🚨 ALERTA: Os dados parecem estar em escala REAL (Max: {max_val}). A função avaliar_modelo espera LOG.")


In [ ]:
#🚩
# >>>Avaliar modelo e registrar métricas para comparar no final 


# Avaliar a modelo
metrics = avaliar_modelo(y_val_log, y_pred_log, nome_modelo="HalvingGridSearchCV + RF (Tuned)")

#------------------------------------------------------/produção
# Registrar Métricas
registrar_metricas(
    modelo_nome="RandomForest (HalvingGridSearchCV)",
    mae=metrics['mae'],
    rmse=metrics['rmse'],
    wape=metrics['wape'],
    mape=metrics.get('mape', 0), # usa 0 se não tiver mape calculado
    r2=metrics['r2'], 
    notebook="06_hyperparameter_tuning.ipynb",
    etapa="Otimização/Tuning"
)


--- Performance: RandomForest (HalvingGridSearchCV) ---
MAE:  7.08 (Erro médio em unidades)
RMSE: 9.26 (Penaliza erros grandes)
WAPE: 10.39% (Erro percentual ponderado)
MAPE: 12.03% (Erro médio absoluto)
R²:   0.9270 (Aderência/Variância explicada)
------------------------------
✅ Métricas registradas com sucesso em: c:\Users\Sergio\Desktop\_ESTUDO\_portifolio_full\retail-demand-forecasting\data\metrics_history.csv


## 💾 Salvando o Arquivo

In [14]:
joblib.dump(best_model, PATH_DATA)
print(f"\nModelo salvo em {PATH_DATA}. Pronto para a etapa de Avaliação.")



Modelo salvo em ../models/model_final.pkl. Pronto para a etapa de Avaliação.
